In [ ]:
import numpy as np
from gfsupg.solver import CartesianGeometry, FiniteElement1D, Scipy2DFEM
from gfsupg.solver import DeC, DeCSpaceTimeSUPGSolver
from gfsupg.problem import *
from gfsupg.plotting import *
import sympy as sym
import os

import matplotlib.pyplot as plt

In [ ]:

#problem.T_fin = 1.
order=5

FEM1Dx = FiniteElement1D(order-1,"gaussLobatto","gaussLegendre")
FEM1Dy = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
dec = DeC((order+1)//2,order,"gaussLobatto")
# dec = DeC(4,5,"gaussLobatto")


In [ ]:
def two_cells_matrix(ZZ):
    AA = np.zeros((2*(order-1)-1,2*(order-1)+1))
    AA[:order-1,:order] = ZZ[1:,:]
    AA[order-2:,order-1:] = AA[order-2:,order-1:] + ZZ[:-1,:]
    AA[abs(AA)<1e-14]=0.
    return AA

def more_cells_matrix(ZZ,N):
    AA = np.zeros((N*(order-1)-1,N*(order-1)+1))
    #i=0
    AA[:order-1,:order] = ZZ[1:,:]
    for i in range(1,N-1):
        AA[(order-1)*i-1:(order-1)*(i+1),(order-1)*i:(order-1)*(i+1)+1] +=  ZZ
    
    #i=N-1
    AA[(order-1)*(N-1)-1:(order-1)*(N)-1,(order-1)*(N-1):(order-1)*(N)+1]+=ZZ[:-1,:]

    AA[abs(AA)<1e-14]=0.
    return AA


def more_cells_matrix_full(ZZ,N):
    AA = np.zeros((N*(order-1)+1,N*(order-1)+1))
    #i=0
    AA[:order,:order] = ZZ[:,:]
    for i in range(1,N-1):
        AA[(order-1)*i:(order-1)*(i+1)+1,(order-1)*i:(order-1)*(i+1)+1] +=  ZZ
    
    #i=N-1
    AA[(order-1)*(N-1):(order-1)*(N)+1,(order-1)*(N-1):(order-1)*(N)+1]+=ZZ[:,:]

    AA[abs(AA)<1e-14]=0.
    return AA

Dij = two_cells_matrix(FEM1Dx.matrix["deriv_ij"])
Dj = two_cells_matrix(FEM1Dx.matrix["deriv_j"])

N_cell = 6
D_mult_j = more_cells_matrix(FEM1Dx.matrix["deriv_j"],N_cell)
D_mult_ij = more_cells_matrix(FEM1Dx.matrix["deriv_ij"],N_cell)
D_mult_i  = more_cells_matrix(FEM1Dx.matrix["deriv_i"],N_cell)
Mass_mult_full = more_cells_matrix_full(FEM1Dx.matrix["lump_mass"],N_cell)
D_mult_full_j = more_cells_matrix_full(FEM1Dx.matrix["deriv_j"],N_cell)
Z= D_mult_ij - D_mult_j@np.linalg.inv(Mass_mult_full)@D_mult_full_j



In [ ]:
Dij_sym = sym.Matrix(Dij)
VV_Dij = Dij_sym.nullspace()

Dj_sym = sym.Matrix(Dj)
VV_Dj = Dj_sym.nullspace()

print("Kernel Dj")
print(np.array(VV_Dj[0]+VV_Dj[1]).flatten())
print(np.array(VV_Dj[0]-VV_Dj[1]).flatten())
print("Kernel Dij")
print(np.array(VV_Dij[0]+VV_Dij[1]).flatten())
print(np.array(VV_Dij[0]-VV_Dij[1]).flatten())


In [ ]:

kernel_folder = "kernels"
os.makedirs(kernel_folder, exist_ok=True) 

Dij_sym = sym.Matrix(D_mult_ij)
VV_Dij = Dij_sym.nullspace()

Dj_sym = sym.Matrix(D_mult_j)
VV_Dj = Dj_sym.nullspace()

Z_sym = sym.Matrix(Z)
VV_Z = Z_sym.nullspace()

def get_kernel_vectors(null_space):
    a = np.array(null_space[0]+null_space[1]).flatten()
    b = np.array(null_space[0]-null_space[1]).flatten()
    return a,b

def plot_vector(u, N_cell, FEM1D, null_space_name):
    uu = np.zeros((FEM1D.degree+1,N_cell))
    xx = np.zeros((FEM1D.degree+1,N_cell))
    x_edge = np.linspace(0,1,N_cell+1)
    dx = x_edge[1]-x_edge[0]
    for i in range(N_cell):
        uu[:,i] = u[i*FEM1D.degree:(i+1)*FEM1D.degree+1]
        xx[:,i] = x_edge[i]+dx*FEM1D.nodes
        poly=np.polyfit(xx[:,i],uu[:,i],FEM1D.degree)
        xx_local = np.linspace(x_edge[i],x_edge[i+1],100)
        plt.plot(xx_local,np.polyval(poly,xx_local))
    plt.plot(xx,uu,'o')
    plt.tight_layout()
    plt.savefig(os.path.join(kernel_folder,f"Kernel_{null_space_name}_deg_{FEM1D.degree}_N_{N_cell}.pdf"))

def plot_kernel(null_space, null_space_name):
    plt.figure()
    a,b = get_kernel_vectors(null_space)
    plot_vector(a,N_cell,FEM1Dx,null_space_name)
    plot_vector(b,N_cell,FEM1Dx,null_space_name)


print("Kernel Dj")
plot_kernel(VV_Dj,"Dj")
plt.title("Kernel Dj")
plot_kernel(VV_Dij,"Dij")
plt.title("Kernel Dij")
plot_kernel(VV_Z,"Z")
plt.title("Kernel Z")

# print(np.array(VV_Dj[0]+VV_Dj[1]).flatten())
# print(np.array(VV_Dj[0]-VV_Dj[1]).flatten())
# print("Kernel Dij")
# print(np.array(VV_Dij[0]+VV_Dij[1]).flatten())
# print(np.array(VV_Dij[0]-VV_Dij[1]).flatten())
# print("Kernel Z")
# print(np.array(VV_Z[0]+VV_Z[1]).flatten())
# print(np.array(VV_Z[0]-VV_Z[1]).flatten())


### Working on broken space P^p continuous test and P^{p-1} discontinuous trial

In [ ]:
order = 3
nodes_type = "gaussLobatto"

In [ ]:
def two_cells_matrix_broken(ZZ):
    AA = np.zeros((2*(order-1)-1,2*(order-1)))
    AA[:order-1,:order-1] = ZZ[1:,:]
    AA[order-2:,order-1:] = ZZ[:-1,:]
    AA[abs(AA)<1e-14]=0.
    return AA

def more_cells_matrix_broken(ZZ,N):
    if N<1:
        raise ValueError("N cannot be less than 1")
    elif N==1:
        return ZZ[1:-1,:]

    (order,p) = np.shape(ZZ)
    if order!=p+1:
        raise ValueError("Matrix not as expected")

    AA = np.zeros((N*(order-1)-1,N*(order-1)))
    AA[:order-1,:order-1] = ZZ[1:,:]
    for k in range(1, N-1):
        vert_shift = k*p-1
        AA[vert_shift:vert_shift+order,p*k:p*(k+1)] = ZZ
    k = N-1
    vert_shift = (N-1)*p-1
    AA[vert_shift:vert_shift+order-1,p*k:p*(k+1)] = ZZ[:-1,:]
    AA[abs(AA)<1e-14]=0.
    return AA

In [ ]:
from gfsupg.quadr import lagrange_basis, lagrange_basis_deriv, nodes_weights

nodes_test,  weights_test  = nodes_weights(order,   nodes_type)
nodes_trial, weights_trial = nodes_weights(order-1, nodes_type)
nodes_quad, weights_quad   = nodes_weights(order,"gaussLobatto")

# Mij
M = np.zeros((len(nodes_test), len(nodes_trial)))
Di = np.zeros((len(nodes_test), len(nodes_trial)))
for i in range(len(nodes_test)):
    phi_test = lagrange_basis(nodes_test, nodes_quad, i)
    phi_der_test = lagrange_basis_deriv(nodes_test, nodes_quad, i)
    for j in range(len(nodes_trial)):
        phi_trial = lagrange_basis(nodes_trial, nodes_quad, j)

        M[i,j] = np.sum(phi_test*phi_trial*weights_quad)
        Di[i,j] = np.sum(phi_der_test*phi_trial*weights_quad)

Mbig = two_cells_matrix_broken(M)
Di_big = two_cells_matrix_broken(Di)
Di_big2 = more_cells_matrix_broken(Di,2)

print(M)

print(Di_big2-Di_big)

In [ ]:
M_N = more_cells_matrix_broken(M,4)
M_N_sym = sym.Matrix(M_N)
VV = M_N_sym.nullspace()
VV

In [ ]:
Mbig_sym = sym.Matrix(Mbig)
VV = Mbig_sym.nullspace()
Di_big_sym = sym.Matrix(Di_big)
VV_Di = Di_big_sym.nullspace()

In [ ]:
M_sym = sym.Matrix(M)
VM = M_sym.nullspace()
VM

In [ ]:
VV

In [ ]:
VV_Di